# 2 · Look at the data

**CPU-only, runs from committed data.**

Three judgement calls hold this study up, and none of them is visible in a
summary statistic:

1. **Segmentation** — is a "sentence" a sensible unit of reasoning?
2. **Answer extraction** — does the extractor read what the rollout concluded?
3. **Category labelling** — do the labels mean what they say?

This notebook puts all three in front of you, using **randomly selected**
examples (seeded, not cherry-picked). It also re-derives an importance value by
hand so you can check the pipeline arithmetic yourself.

In [1]:
import json, sys
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "code"))
DATA = ROOT / "data"
TAG  = "DeepSeek-R1-Distill-Qwen-14B"

traces  = json.loads((DATA / f"traces_{TAG}.json").read_text())
rollouts = {(r["trace_id"], r["i"]): r
            for r in json.loads((DATA / f"rollouts_main_{TAG}.json").read_text())}
df = pd.read_csv(DATA / f"sentences_{TAG}.csv")
print(f"{len(traces)} traces | {len(rollouts)} swept prefixes | {len(df)} scored sentences")

8 traces | 312 swept prefixes | 254 scored sentences


## 2.1 Segmentation

These models write **outlines**, not prose. Bullets and numbered items are unit
boundaries, and list markers are masked from the sentence-terminator rule —
otherwise units come out as `"Here's a thinking process: 1."` with the *next*
item's number glued onto the previous unit.

In [2]:
rng = np.random.default_rng(0)
t = traces[int(rng.integers(len(traces)))]
print(f"problem : {t['pid']}  (level {t['level']})")
print(f"gold    : {t['gold']}   base trace answer: {t['base_answer']} "
      f"({'correct' if t['base_correct'] else 'incorrect'})")
print(f"trace   : {t['n_sentences']} sentences, {len(t['thinking'])} chars\n")
for s in t["sentences"][:12]:
    print(f"  [{s['index']:3d}] {' '.join(s['text'].split())[:96]}")

problem : test/counting_and_probability/134.json  (level 4)
gold    : 720   base trace answer: 720 (correct)
trace   : 33 sentences, 2522 chars

  [  0] Okay, so I need to figure out how many ways 8 people can sit around a round table, but with the 
  [  1] Hmm, circular permutations can be a bit tricky, but I think I remember some basics.
  [  2] First, when arranging people around a round table, the number of distinct arrangements is usuall
  [  3] where n is the number of people.
  [  4] This is because rotations of the same arrangement are considered identical.
  [  5] So, for 8 people, it would normally be 7!
  [  6] = 5040 ways.
  [  7] But since Pierre, Rosa, and Thomas want to sit together, I need to adjust for that.
  [  8] I think the way to handle this is to treat Pierre, Rosa, and Thomas as a single unit or "block."
  [  9] So, instead of thinking about arranging 8 individuals, I can think about arranging this block al
  [ 10] That effectively reduces the problem to arrangi

### Prefixes are exact slices — verify it

A rollout prompt is literally `thinking[:span.end]`, i.e. the bytes the model
itself produced. Nothing is re-rendered, so no measured effect can be an artefact
of reconstruction.

In [3]:
ok_prefix = all(t["thinking"].startswith(t["thinking"][: s["end"]])
                for s in t["sentences"])
tiles = "".join(t["thinking"][s["start"]:s["end"]] for s in t["sentences"]) == t["thinking"]
print(f"every prefix is a literal prefix of the trace : {ok_prefix}")
print(f"spans tile the trace exactly, no gaps         : {tiles}")

every prefix is a literal prefix of the trace : True
spans tile the trace exactly, no gaps         : True


## 2.2 Re-derive an importance value by hand

Don't take the pipeline's word for it. Pull the raw answer lists for prefixes
`i-1` and `i`, and compute the Jeffreys-smoothed KL yourself.

In [4]:
row = df[(df.kl_resampling > 0.05) & (df.kl_resampling < 0.3)].iloc[3]
tid, i = row.trace_id, int(row["index"])

a_before = rollouts[(tid, i-1)]["answers"]
a_after  = rollouts[(tid, i)]["answers"]

LAM = 0.5                                   # Jeffreys, as in anchors/importance.py
cb = Counter(a if a is not None else "<none>" for a in a_before)
ca = Counter(a if a is not None else "<none>" for a in a_after)
support = sorted(set(cb) | set(ca))

p = np.array([cb[s] + LAM for s in support]); p /= p.sum()
q = np.array([ca[s] + LAM for s in support]); q /= q.sum()
kl_hand = float((p * np.log(p / q)).sum())

print(f"trace {tid}, sentence {i}")
print(f"  rollouts  : {len(a_before)} before, {len(a_after)} after")
print(f"  support   : {support}")
print(f"  before    : {[cb[s] for s in support]}")
print(f"  after     : {[ca[s] for s in support]}")
print(f"\n  hand-computed KL : {kl_hand:.6f}")
print(f"  pipeline says    : {row.kl_resampling:.6f}")
print(f"  MATCH            : {abs(kl_hand - row.kl_resampling) < 1e-9}")
print(f"\n  sentence: {' '.join(str(row.text).split())[:110]}")

trace test/intermediate_algebra/1388.json#0, sentence 0
  rollouts  : 64 before, 64 after
  support   : ['-2', '-2,1', '1']
  before    : [2, 18, 44]
  after     : [2, 6, 56]

  hand-computed KL : 0.133221
  pipeline says    : 0.133221
  MATCH            : True

  sentence: Okay, so I have this functional equation: f(x) + f(y) = f(x + y) - xy - 1 for all real numbers x and y.


## 2.3 What a rollout actually looks like

The measure rests on these continuations being sensible. Two randomly chosen
ones, with the extracted answer alongside so you can check the extractor.

In [5]:
rec = rollouts[list(rollouts)[int(rng.integers(len(rollouts)))]]
tr  = next(x for x in traces if x["trace_id"] == rec["trace_id"])
print(f"trace {rec['trace_id']}, prefix ends after sentence {rec['i']} "
      f"of {tr['n_sentences']}\n")
if rec["i"] >= 0:
    print("PREFIX ENDS WITH:")
    print("   ", " ".join(tr["sentences"][rec["i"]]["text"].split())[:150], "\n")
for j, comp in enumerate(rec["sample_completions"][:2]):
    print(f"--- continuation {j+1} | extracted answer: {rec['answers'][j]!r} "
          f"| gold: {tr['gold']!r}")
    print("   ", " ".join(comp.split())[:420], "\n")

trace test/algebra/661.json#0, prefix ends after sentence 115 of 117

PREFIX ENDS WITH:
    So, in conclusion, the fewest number of miles Suzanne can walk in February is 36 miles. 

--- continuation 1 | extracted answer: '36' | gold: '36'
    </think> Suzanne walks four miles every third day. To find the fewest number of miles she can walk in February, we consider the shortest February with 28 days. She walks on every third day, which are days 3, 6, 9, ..., up to day 27. This results in 9 walking days. Calculating the total miles: \[ 9 \text{ days} \times 4 \text{ miles/day} = 36 \text{ miles} \] Thus, the fewest number of miles Suzanne can walk in Februa 

--- continuation 2 | extracted answer: '36' | gold: '36'
    </think> Suzanne walks four miles every third day. February has 28 days in a common year and 29 in a leap year. To find the fewest miles, we consider the shorter month, 28 days. She walks every third day, which are days 3, 6, 9, ..., up to the maximum day in February. Divi

## 2.4 How the answer distribution actually moves

Walk one trace's answer distribution along its swept prefixes. This is the object
the whole measure is built on — and you can see the convergence that motivated
the positional hypothesis in the first place.

In [6]:
tid = traces[1]["trace_id"]
n   = next(t for t in traces if t["trace_id"] == tid)["n_sentences"]
idx = sorted(i for (tt, i) in rollouts if tt == tid)
print(f"{tid}  ({n} sentences)\n")
print(f"{'prefix':>7} {'pos':>5}  {'entropy':>7}  answer distribution")
for i in idx:
    ans = rollouts[(tid, i)]["answers"]
    c = Counter(a if a is not None else "<none>" for a in ans)
    tot = sum(c.values())
    pr = np.array(list(c.values())) / tot
    H = float(-(pr * np.log(pr)).sum())
    top = ", ".join(f"{k}:{v}" for k, v in c.most_common(3))
    pos = "  --  " if i < 0 else f"{(i+1)/n:5.2f}"
    print(f"{i:>7} {pos}  {H:7.3f}  {top}")

test/intermediate_algebra/1388.json#0  (170 sentences)

 prefix   pos  entropy  answer distribution
     -1   --      0.723  1:44, -2,1:18, -2:2
      0  0.01    0.447  1:56, -2,1:6, -2:2
      1  0.01    0.433  1:54, -2,1:10
      2  0.02    0.234  1:60, -2,1:4
      3  0.02    0.643  1:42, -2,1:22
     23  0.14    0.512  1:54, -2,1:8, -2:2
     24  0.15    0.512  1:54, -2,1:8, 2:2
     25  0.15    0.483  1:52, -2,1:12
     26  0.16    0.377  1:56, -2,1:8
     27  0.16    0.377  1:56, -2,1:8
     35  0.21    0.311  1:58, -2,1:6
     36  0.22    0.525  1:50, -2,1:14
     37  0.22    0.311  1:58, -2,1:6
     38  0.23    0.377  1:56, -2,1:8
     39  0.24    0.447  1:56, -2,1:6, -2:2
     54  0.32    0.377  1:56, -2,1:8
     55  0.33    0.525  1:50, -2,1:14
     56  0.34    0.463  1:56, -2,1:4, -2:4
     57  0.34    0.562  1:48, -2,1:16
     58  0.35    0.311  1:58, -2,1:6
     84  0.50    0.594  1:46, -2,1:18
     85  0.51    0.643  1:42, -2,1:22
     86  0.51    0.594  1:46, -2,1:18
   

**Read the entropy column top to bottom.** It collapses as the trace
proceeds — the convergence is real. My hypothesis was that this convergence
drives the importance measure. Notebook 1 shows it does not; what drives it is
*how much entropy was there to move*, which is a different thing.

## 2.5 The control I withdrew

The filler arm replaced a sentence with a bland continuation, holding position
fixed and removing content. A **pre-registered** check (P8) required the filler
to sit at least 2× closer to the model's own sentences than a deliberately alien
sentence, measured by the model's own log-probabilities.

In [7]:
f = json.loads((DATA / f"filler_indistribution_{TAG}.json").read_text())
print(f"slots scored: {f['n_slots']}\n")
print(f"  the model's own sentence  {f['mean_logp_real']:+.3f} nats/token")
print(f"  my filler                 {f['mean_logp_filler']:+.3f}   "
      f"({f['filler_gap_in_sd']:+.2f} sd from real)")
print(f"  deliberately alien text   {f['mean_logp_alien']:+.3f}   "
      f"({f['alien_gap_in_sd']:+.2f} sd from real)")
ratio = f["alien_gap_in_sd"] / f["filler_gap_in_sd"]
print(f"\n  ratio required by P8 : >= 2.00")
print(f"  ratio measured       : {ratio:.2f}      --> FAILED")
print(f"  filler more likely than the real sentence in "
      f"{f['frac_filler_more_likely_than_real']:.0%} of slots")
print("\nThe filler was an intrusion, so the arm measured disruption rather than")
print("'same position, no content'. Its result is withdrawn, as pre-registered.")

slots scored: 120

  the model's own sentence  -0.529 nats/token
  my filler                 -4.720   (-7.61 sd from real)
  deliberately alien text   -6.959   (-11.68 sd from real)

  ratio required by P8 : >= 2.00
  ratio measured       : 1.53      --> FAILED
  filler more likely than the real sentence in 0% of slots

The filler was an intrusion, so the arm measured disruption rather than
'same position, no content'. Its result is withdrawn, as pre-registered.


**Why this is the most useful cell in the notebook.** The filler arm's
headline number — real sentences beat their fillers in only 32% of slots — reads
as strong evidence that the measure ignores content, and points the same way as
the paraphrase result. It is evidence of nothing. The rule written down in
advance is what made discarding it automatic rather than a judgement call I was
motivated to get wrong.